In [10]:
import numpy as np
from scipy import stats

from Registration.uti import load_registration_results


selected_organs = [
    "brain",
    "skull",
    "heart",
    "aorta",
    "trachea",
    "esophagus",
    "liver",
    "spleen",
    "pancreas",
    "stomach",
    "kidney_left",
    "kidney_right",
    "colon",
    "urinary_bladder",
    "prostate",
    "spinal_cord",
    "vertebrae_L3",
]

s = 4500


a = fr"C:\Users\Sam\Downloads\pet_reg_results\ctsmoothness_l{s}_k10_mar3000_gam1.0.txt"
b = fr"C:\Users\Sam\Downloads\pet_reg_results\baseline_l{s}_k10.txt"


masks_names, dice_before_lists, \
    dice_after_lists_0, tre_before_lists, tre_after_lists_0 = \
        load_registration_results(a)

masks_names, dice_before_lists, \
    dice_after_lists_1, tre_before_lists, tre_after_lists_1 = \
        load_registration_results(b)

selected_indices = [masks_names.index(organ) for organ in selected_organs]




import numpy as np
import ast
from scipy import stats

def _to_float_array(x):
    # If it's a string like "[0.1, 0.2, ...]" parse it
    if isinstance(x, str):
        x = ast.literal_eval(x)

    arr = np.asarray(x, dtype=float).ravel()

    # Remove NaN/inf if any
    arr = arr[np.isfinite(arr)]
    return arr

def print_p(listA, listB, organ_name):
    model_A = _to_float_array(listA)
    model_B = _to_float_array(listB)

    if model_A.shape != model_B.shape:
        raise ValueError(f"{organ_name}: shape mismatch A{model_A.shape} vs B{model_B.shape}")

    # Paired t-test
    t_stat, p_value = stats.ttest_rel(model_A, model_B)

    # Wilcoxon (more robust for non-normal metrics)
    try:
        w_stat, p_w = stats.wilcoxon(model_A, model_B, zero_method="wilcox")
    except ValueError:
        # can happen if all differences are zero
        w_stat, p_w = np.nan, 1.0

    diff = model_A - model_B
    mean_diff = diff.mean()
    std_diff = diff.std(ddof=1) if diff.size > 1 else np.nan
    cohen_d = mean_diff / std_diff if (std_diff not in [0, np.nan] and std_diff != 0) else np.nan

    print(f"\nOrgan: {organ_name}")
    print(f"N = {len(model_A)}")
    print(f"Mean(A) = {model_A.mean():.4f} ± {model_A.std(ddof=1):.4f}")
    print(f"Mean(B) = {model_B.mean():.4f} ± {model_B.std(ddof=1):.4f}")
    print(f"Mean(A-B) = {mean_diff:.4f}")

    print("Paired t-test:    t = {:.4f}, p = {:.4e}".format(t_stat, p_value))
    print("Wilcoxon test:    W = {}, p = {:.4e}".format(w_stat, p_w))
    print("Effect size (paired Cohen's d) = {:.4f}".format(cohen_d))


for organ in selected_organs:
    index = masks_names.index(organ)
    # usage
    print_p(dice_after_lists_0[index], dice_after_lists_1[index], organ)

print("********************************")
for organ in selected_organs:
    index = masks_names.index(organ)
    # usage
    print_p(tre_after_lists_0[index], tre_after_lists_1[index], organ)



Organ: brain
N = 80
Mean(A) = 0.8243 ± 0.0889
Mean(B) = 0.8737 ± 0.0755
Mean(A-B) = -0.0494
Paired t-test:    t = -8.9888, p = 1.0231e-13
Wilcoxon test:    W = 186.0, p = 6.0752e-12
Effect size (paired Cohen's d) = -1.0050

Organ: skull
N = 80
Mean(A) = 0.4866 ± 0.1383
Mean(B) = 0.5823 ± 0.1369
Mean(A-B) = -0.0957
Paired t-test:    t = -12.8004, p = 5.8898e-21
Wilcoxon test:    W = 78.0, p = 1.4048e-13
Effect size (paired Cohen's d) = -1.4311

Organ: heart
N = 80
Mean(A) = 0.7219 ± 0.0987
Mean(B) = 0.7888 ± 0.0795
Mean(A-B) = -0.0669
Paired t-test:    t = -9.2667, p = 2.9342e-14
Wilcoxon test:    W = 198.0, p = 9.0838e-12
Effect size (paired Cohen's d) = -1.0361

Organ: aorta
N = 80
Mean(A) = 0.5466 ± 0.2230
Mean(B) = 0.7354 ± 0.1325
Mean(A-B) = -0.1888
Paired t-test:    t = -12.1463, p = 9.3450e-20
Wilcoxon test:    W = 66.0, p = 9.0944e-14
Effect size (paired Cohen's d) = -1.3580

Organ: trachea
N = 80
Mean(A) = 0.3671 ± 0.2237
Mean(B) = 0.5938 ± 0.1764
Mean(A-B) = -0.2267
Paired t-

In [16]:

_list = ['baseline_l5500_k10', 'ctsmoothness_l5500_k10_mar3000_gam2.0']
for item in _list:
    path = fr"C:\Users\Sam\Downloads\pet_reg_results_whole_body\{item}.txt"

    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    mi  = np.array([float(x) for x in lines[0].split()])
    ncc = np.array([float(x) for x in lines[1].split()])


    print(mi.mean(), ncc.mean())




0.5630277125 0.7034727625
0.6192400749999999 0.7218136625


In [28]:
import numpy as np
from scipy import stats
__list = [4000, 4500, 5000, 5500, 6000, 6500, 7000]
_list = ['baseline_l4000_k10', 'ctsmoothness_l4000_k10_mar3000_gam2.0']

def read_mi_ncc(path):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    mi  = np.array([float(x) for x in lines[0].split()], dtype=np.float64)
    ncc = np.array([float(x) for x in lines[1].split()], dtype=np.float64)
    return mi, ncc

# read both methods
mi0, ncc0 = read_mi_ncc(fr"C:\Users\Sam\Downloads\pet_reg_results_whole_body\{_list[0]}.txt")
mi1, ncc1 = read_mi_ncc(fr"C:\Users\Sam\Downloads\pet_reg_results_whole_body\{_list[1]}.txt")

# sanity check
assert mi0.shape == mi1.shape, f"MI length mismatch: {mi0.shape} vs {mi1.shape}"
assert ncc0.shape == ncc1.shape, f"NCC length mismatch: {ncc0.shape} vs {ncc1.shape}"

print("Means:")
print(_list[0], "MI mean", mi0.mean(), "NCC mean", ncc0.mean())
print(_list[1], "MI mean", mi1.mean(), "NCC mean", ncc1.mean())

# paired t-tests
t_mi,  p_mi  = stats.ttest_rel(mi0,  mi1,  nan_policy="omit")
t_ncc, p_ncc = stats.ttest_rel(ncc0, ncc1, nan_policy="omit")

print("\nPaired t-test (baseline vs proposed):")
print(f"MI : t = {t_mi:.4f},  p = {p_mi:.4g}")
print(f"NCC: t = {t_ncc:.4f}, p = {p_ncc:.4g}")



def mean_ci(x, alpha=0.05):
    """
    95% CI for the mean of x using Student-t.
    Returns: mean, (ci_low, ci_high), n
    """
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        raise ValueError("Need at least 2 samples for CI.")
    m = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    return m, (m - tcrit*se, m + tcrit*se), n

# baseline
mi0_mean, mi0_ci, n0 = mean_ci(mi0)
ncc0_mean, ncc0_ci, _ = mean_ci(ncc0)

# proposed
mi1_mean, mi1_ci, n1 = mean_ci(mi1)
ncc1_mean, ncc1_ci, _ = mean_ci(ncc1)

print("95% CI for mean (each method):")
print(f"{_list[0]}  MI : mean={mi0_mean:.6f}  CI=[{mi0_ci[0]:.6f}, {mi0_ci[1]:.6f}]  n={n0}")
print(f"{_list[0]}  NCC: mean={ncc0_mean:.6f} CI=[{ncc0_ci[0]:.6f}, {ncc0_ci[1]:.6f}]  n={n0}")
print(f"{_list[1]}  MI : mean={mi1_mean:.6f}  CI=[{mi1_ci[0]:.6f}, {mi1_ci[1]:.6f}]  n={n1}")
print(f"{_list[1]}  NCC: mean={ncc1_mean:.6f} CI=[{ncc1_ci[0]:.6f}, {ncc1_ci[1]:.6f}]  n={n1}")

Means:
baseline_l4000_k10 MI mean 0.59832985 NCC mean 0.709170575
ctsmoothness_l4000_k10_mar3000_gam2.0 MI mean 0.6115843625 NCC mean 0.7157250625

Paired t-test (baseline vs proposed):
MI : t = -3.6545,  p = 0.0004621
NCC: t = -4.7064, p = 1.057e-05
95% CI for mean (each method):
baseline_l4000_k10  MI : mean=0.598330  CI=[0.574482, 0.622178]  n=80
baseline_l4000_k10  NCC: mean=0.709171 CI=[0.700624, 0.717717]  n=80
ctsmoothness_l4000_k10_mar3000_gam2.0  MI : mean=0.611584  CI=[0.585327, 0.637842]  n=80
ctsmoothness_l4000_k10_mar3000_gam2.0  NCC: mean=0.715725 CI=[0.706967, 0.724483]  n=80


In [34]:
import numpy as np
from scipy import stats
import ast
def load_registration_results(filename):
    """
    Load registration results from text file.

    Returns:
        masks_names,
        dice_before_lists,
        dice_after_lists,
        tre_before_lists,
        tre_after_lists
    """

    def parse_list_line(line):
        parts = line.strip().split(";")
        parsed = []
        for p in parts:
            p = p.strip()

            # try to interpret as python literal (list, float, etc.)
            try:
                parsed.append(ast.literal_eval(p))
            except Exception:
                parsed.append(p)

        return parsed

    with open(filename, "r") as f:
        lines = f.readlines()

    masks_names = parse_list_line(lines[0])
    dice_before_lists = parse_list_line(lines[1])
    dice_after_lists = parse_list_line(lines[2])
    tre_before_lists = parse_list_line(lines[3])
    tre_after_lists = parse_list_line(lines[4])

    return (
        masks_names,
        dice_before_lists,
        dice_after_lists,
        tre_before_lists,
        tre_after_lists,
    )

hyperparameters = [4000, 4500, 5000, 5500, 6000, 6500, 7000]
base_dir = r"C:\Users\Sam\Downloads\pet_reg_results_whole_body"
base_dir_mask = r"C:\Users\Sam\Downloads\pet_reg_results"

def mean_std(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan, 0
    m = x.mean()
    s = x.std(ddof=1) if n > 1 else 0.0   # sample std
    return m, s, n


for L in hyperparameters:
    baseline_name = f"baseline_l{L}_k10"
    smooth_name   = f"ctsmoothness_l{L}_k10_mar3000_gam2.0"

    path0 = fr"{base_dir}\{baseline_name}.txt"
    path1 = fr"{base_dir}\{smooth_name}.txt"
    path0_baseline = fr"{base_dir_mask}\{baseline_name}.txt"
    path1_smooth = fr"{base_dir_mask}\{smooth_name}.txt"


    _, _, dice_after_lists_0, _, tre_after_lists_0 = load_registration_results(path0_baseline)
    _, _, dice_after_lists_1, _, tre_after_lists_1 = load_registration_results(path1_smooth)

    # print(np.asarray(dice_after_lists_0, dtype=float))

    dice_baseline = np.asarray(dice_after_lists_0, dtype=float).mean(axis=0)
    dice_proposed = np.asarray(dice_after_lists_1, dtype=float).mean(axis=0)
    print(dice_baseline.shape)# this is (80, )


    tre_baseline = np.asarray(tre_after_lists_0, dtype=float).mean(axis=0)
    tre_proposed = np.asarray(tre_after_lists_1, dtype=float).mean(axis=0)
    # print(tre.shape)# this is (80, )


    mi0, ncc0 = read_mi_ncc(path0)
    mi1, ncc1 = read_mi_ncc(path1)

    assert mi0.shape == mi1.shape, f"[L={L}] MI length mismatch: {mi0.shape} vs {mi1.shape}"
    assert ncc0.shape == ncc1.shape, f"[L={L}] NCC length mismatch: {ncc0.shape} vs {ncc1.shape}"

    # paired t-test
    t_mi,  p_mi  = stats.ttest_rel(mi0,  mi1,  nan_policy="omit")
    t_ncc, p_ncc = stats.ttest_rel(ncc0, ncc1, nan_policy="omit")

    mi0_mean, mi0_std, n0 = mean_std(mi0)
    ncc0_mean, ncc0_std, _ = mean_std(ncc0)
    mi1_mean, mi1_std, n1 = mean_std(mi1)
    ncc1_mean, ncc1_std, _ = mean_std(ncc1)

    # ----- after you compute mi0, mi1, dice_baseline, dice_proposed, tre_baseline, tre_proposed -----

    # sanity checks for paired tests
    assert dice_baseline.shape == dice_proposed.shape, f"[L={L}] Dice length mismatch: {dice_baseline.shape} vs {dice_proposed.shape}"
    assert tre_baseline.shape == tre_proposed.shape, f"[L={L}] TRE length mismatch: {tre_baseline.shape} vs {tre_proposed.shape}"

    # Paired t-tests (baseline vs proposed)
    t_mi,   p_mi   = stats.ttest_rel(mi0,            mi1,            nan_policy="omit")
    t_ncc,  p_ncc  = stats.ttest_rel(ncc0,           ncc1,           nan_policy="omit")
    t_dice, p_dice = stats.ttest_rel(dice_baseline,  dice_proposed,  nan_policy="omit")
    t_tre,  p_tre  = stats.ttest_rel(tre_baseline,   tre_proposed,   nan_policy="omit")

    # Mean ± std
    mi0_mean,   mi0_std,   mi_n0   = mean_std(mi0)
    mi1_mean,   mi1_std,   mi_n1   = mean_std(mi1)

    ncc0_mean,  ncc0_std,  ncc_n0  = mean_std(ncc0)
    ncc1_mean,  ncc1_std,  ncc_n1  = mean_std(ncc1)

    dice0_mean, dice0_std, dice_n0 = mean_std(dice_baseline)
    dice1_mean, dice1_std, dice_n1 = mean_std(dice_proposed)

    tre0_mean,  tre0_std,  tre_n0  = mean_std(tre_baseline)
    tre1_mean,  tre1_std,  tre_n1  = mean_std(tre_proposed)

    # Pretty print
    print(f"\n{'='*70}")
    print(f"L = {L}")
    print(f"{'='*70}")

    print(f"MI   baseline: {mi0_mean:.6f} ± {mi0_std:.6f} (n={mi_n0})")
    print(f"MI   proposed: {mi1_mean:.6f} ± {mi1_std:.6f} (n={mi_n1})")
    print(f"MI   paired t-test: t = {t_mi:.6f}, p = {p_mi:.6e}")

    print(f"NCC  baseline: {ncc0_mean:.6f} ± {ncc0_std:.6f} (n={ncc_n0})")
    print(f"NCC  proposed: {ncc1_mean:.6f} ± {ncc1_std:.6f} (n={ncc_n1})")
    print(f"NCC  paired t-test: t = {t_ncc:.6f}, p = {p_ncc:.6e}")

    print(f"Dice baseline: {dice0_mean:.6f} ± {dice0_std:.6f} (n={dice_n0})")
    print(f"Dice proposed: {dice1_mean:.6f} ± {dice1_std:.6f} (n={dice_n1})")
    print(f"Dice paired t-test: t = {t_dice:.6f}, p = {p_dice:.6e}")

    print(f"TRE  baseline: {tre0_mean:.6f} ± {tre0_std:.6f} (n={tre_n0})")
    print(f"TRE  proposed: {tre1_mean:.6f} ± {tre1_std:.6f} (n={tre_n1})")
    print(f"TRE  paired t-test: t = {t_tre:.6f}, p = {p_tre:.6e}")









(80,)

L = 4000
MI   baseline: 0.598330 ± 0.107164 (n=80)
MI   proposed: 0.611584 ± 0.117990 (n=80)
MI   paired t-test: t = -3.654534, p = 4.620617e-04
NCC  baseline: 0.709171 ± 0.038406 (n=80)
NCC  proposed: 0.715725 ± 0.039356 (n=80)
NCC  paired t-test: t = -4.706426, p = 1.057253e-05
Dice baseline: 0.519549 ± 0.109282 (n=80)
Dice proposed: 0.481564 ± 0.142451 (n=80)
Dice paired t-test: t = 5.253527, p = 1.232831e-06
TRE  baseline: 7.996379 ± 3.942464 (n=80)
TRE  proposed: 11.077305 ± 7.182182 (n=80)
TRE  paired t-test: t = -6.305867, p = 1.532836e-08
(80,)

L = 4500
MI   baseline: 0.598644 ± 0.110257 (n=80)
MI   proposed: 0.646774 ± 0.099811 (n=80)
MI   paired t-test: t = -7.111474, p = 4.570610e-10
NCC  baseline: 0.706387 ± 0.039830 (n=80)
NCC  proposed: 0.729394 ± 0.038609 (n=80)
NCC  paired t-test: t = -9.964318, p = 1.293237e-15
Dice baseline: 0.518082 ± 0.115635 (n=80)
Dice proposed: 0.575390 ± 0.106581 (n=80)
Dice paired t-test: t = -8.953954, p = 1.196561e-13
TRE  baseline: 8